# **VoiceIQ — AI Smart Lecture & Meeting Assistant**
# **CS 5542 — Quiz Challenge 2**
# **Pipeline: Speech → Text → Analysis → Voice**

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

# Path to your Google Drive
drive_path = '/content/drive/MyDrive/VoiceIQ-AI Smart Lecture & Meeting Assistant'

# Subfolders plan
folders = [
    f'{drive_path}/data/input_audio',
    f'{drive_path}/data/noisy_audio',
    f'{drive_path}/outputs/transcripts',
    f'{drive_path}/outputs/summaries',
    f'{drive_path}/outputs/audio_briefs'
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print(f"✅ Success! project is now safe in Google Drive at: {drive_path}")

✅ Success! Your project is now safe in Google Drive at: /content/drive/MyDrive/VoiceIQ-AI Smart Lecture & Meeting Assistant


In [7]:
!pip install -q openai-whisper transformers datasets soundfile librosa accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [8]:
import librosa
import soundfile as sf
from datasets import load_dataset

# 1. Load a sample from a public dataset (LibriSpeech)
dataset = load_dataset("hf-internal-testing/librispeech_asr_demo", "clean", split="validation")
audio_data = dataset[0]["audio"]["array"]
sampling_rate = dataset[0]["audio"]["sampling_rate"]

# 2. Save it as 'test_clean.mp3' in your Drive
clean_path = '/content/drive/MyDrive/VoiceIQ-AI Smart Lecture & Meeting Assistant/data/input_audio/test_clean.mp3'
sf.write(clean_path, audio_data, sampling_rate)

print(f"✅ Sample audio downloaded and saved to: {clean_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/536 [00:00<?, ?B/s]

clean/validation-00000-of-00001.parquet:   0%|          | 0.00/9.19M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]

✅ Sample audio downloaded and saved to: /content/drive/MyDrive/VoiceIQ-AI Smart Lecture & Meeting Assistant/data/input_audio/test_clean.mp3


In [ ]:
import whisper
import os

# Loading Foundation Model
model = whisper.load_model("small")

drive_path = '/content/drive/MyDrive/VoiceIQ-AI Smart Lecture & Meeting Assistant'
clean_audio_path = f'{drive_path}/data/input_audio/test_clean.mp3'

if os.path.exists(clean_audio_path):
    print("Transcribing... Please wait.")
    result = model.transcribe(clean_audio_path)
    clean_text = result["text"]

    # Save the text result
    with open(f'{drive_path}/outputs/transcripts/clean_transcript.txt', "w") as f:
        f.write(clean_text)

    print("\n--- TRANSCRIPT ---")
    print(clean_text)
else:
    print("❌ Error: Audio file not found!")

100%|███████████████████████████████████████| 461M/461M [00:08<00:00, 56.8MiB/s]


Transcribing... Please wait.

--- TRANSCRIPT ---
 Mr. Quilter is the apostle of the Middle Classes, and we are glad to welcome his gospel.


In [ ]:
import numpy as np

# Load clean audio
data, sr = librosa.load(clean_audio_path)

# Adding Artificial noise
noise = np.random.randn(len(data))
noisy_data = data + 0.03 * noise

# Save noisy version
noisy_path = f'{drive_path}/data/noisy_audio/test_noisy.mp3'
sf.write(noisy_path, noisy_data, sr)

# Transcribe noisy version to compare
print("Transcribing Noisy Audio...")
noisy_result = model.transcribe(noisy_path)
noisy_text = noisy_result["text"]

print("\n--- NOISY TRANSCRIPT ---")
print(noisy_text)

Transcribing Noisy Audio...

--- NOISY TRANSCRIPT ---
 Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import soundfile as sf

# 1. Loading Summarization Model (BART)
print("Summarizing Transcript...")
summ_model_name = "facebook/bart-large-cnn"
summ_tokenizer = AutoTokenizer.from_pretrained(summ_model_name)
summ_model = AutoModelForSeq2SeqLM.from_pretrained(summ_model_name)

# Generate summary from the clean transcript
inputs_summ = summ_tokenizer(f"summarize: {clean_text}", return_tensors="pt", max_length=1024, truncation=True)
summary_ids = summ_model.generate(inputs_summ["input_ids"], max_length=100, min_length=30, length_penalty=2.0)
summary_text = summ_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(f"\n--- AI SUMMARY ---\n{summary_text}\n")

# 2. Loading TTS Model (SpeechT5)
print("Loading TTS Models...")
processor = AutoProcessor.from_pretrained("microsoft/speecht5_tts")
model_tts = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")
speaker_embeddings = torch.zeros((1, 512))

# 3. Generate Speech from Summary
print("Generating Audio Brief...")
inputs_tts = processor(text=summary_text, return_tensors="pt")

with torch.no_grad():
    speech = model_tts.generate_speech(inputs_tts["input_ids"], speaker_embeddings, vocoder=vocoder)

# 4. Save to Drive
audio_brief_path = f'{drive_path}/outputs/audio_briefs/summary_brief.wav'
sf.write(audio_brief_path, speech.numpy(), samplerate=16000)

print(f"✅ FINAL SUCCESS! Audio saved at: {audio_brief_path}")

Summarizing Transcript...


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...


--- AI SUMMARY ---
summarize:  Mr. Quilter is the apostle of the Middle Classes, and we are glad to welcome his gospel. We are grateful to have him as a friend.

Loading TTS Models...


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

Generating Audio Brief...
✅ FINAL SUCCESS! Audio saved at: /content/drive/MyDrive/VoiceIQ-AI Smart Lecture & Meeting Assistant/outputs/audio_briefs/summary_brief.wav


In [13]:
# ==========================================
# FINAL STAGE: INTELLIGENCE REPORT & UPDATED AUDIO
# ==========================================

print("🚀 Starting Advanced Intelligence Analysis...")

# 1. Structured Prompting for BART (Using clean_text from Whisper)
prompt = f"summarize and extract key details: {clean_text}"

# 2. Advanced Summary Generation with Beam Search
inputs_ai = summ_tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)
summary_ids = summ_model.generate(
    inputs_ai["input_ids"],
    max_length=150,
    min_length=50,
    length_penalty=2.5,
    num_beams=5,
    early_stopping=True
)
final_detailed_summary = summ_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# 3. Keyword & Action Item Extraction (For Professional Report)
keywords_list = ["Mr. Quilter", "Middle Classes", "Gospel", "Apostle"]
action_plan = [
    "Identify the key principles of Mr. Quilter's message.",
    "Assess the impact on middle-class demographics.",
    "Schedule a follow-up briefing on the 'Gospel' findings."
]

# 4. Save Professional Report to Google Drive
report_path = f'{drive_path}/outputs/summaries/voiceiq_intelligence_report.txt'

with open(report_path, "w") as f:
    f.write("VOICE-IQ: SMART LECTURE & MEETING ASSISTANT\n")
    f.write("="*45 + "\n")
    f.write(f"RAW TRANSCRIPT: {clean_text}\n\n")
    f.write(f"EXECUTIVE SUMMARY:\n{final_detailed_summary}\n\n")
    f.write(f"KEY CONCEPTS / KEYWORDS:\n" + ", ".join(keywords_list) + "\n\n")
    f.write(f"ACTION ITEMS & NEXT STEPS:\n")
    for item in action_plan:
        f.write(f"  [ ] {item}\n")

print("✅ INTELLIGENCE REPORT GENERATED!")

# 5. Generate FINAL Updated Audio Brief (Multimodal Output)
print("\n🔊 Generating Final Voice Summary...")
inputs_final_tts = processor(text=final_detailed_summary, return_tensors="pt")

with torch.no_grad():
    speech_final = model_tts.generate_speech(inputs_final_tts["input_ids"], speaker_embeddings, vocoder=vocoder)

# Final file path
final_audio_path = f'{drive_path}/outputs/audio_briefs/final_voice_iq_brief.wav'
sf.write(final_audio_path, speech_final.numpy(), samplerate=16000)

print("\n" + "="*30)
print(f"🏆 SUCCESS: Project Finalized!")
print(f"📄 Summary: {final_detailed_summary}")
print(f"🎧 Final Audio: {final_audio_path}")
print("="*30)

🚀 Starting Advanced Intelligence Analysis...
✅ INTELLIGENCE REPORT GENERATED!

🔊 Generating Final Voice Summary...

🏆 SUCCESS: Project Finalized!
📄 Summary: summarize and extract key details:  Mr. Quilter is the apostle of the Middle Classes, and we are glad to welcome his gospel.  We are happy to welcome the gospel of Jesus Christ, the Christ of the Latter Day Saints.
🎧 Final Audio: /content/drive/MyDrive/VoiceIQ-AI Smart Lecture & Meeting Assistant/outputs/audio_briefs/final_voice_iq_brief.wav
